#### Antes de empezar se deben hacer los siguientes cambios : 
####   1. Crear cuenta Databricks Free Edition y completar "Verify identity".
####   2. Correr 00_entrenar_modelo_local.py en local para generar, modelo.pkl y datos.csv.
####   3. Crear una carpeta en Workspace y subir modelo.pkl y datos.csv.



#### Verificar compute y crear el schema "ml" dentro del catalogo "workspace".
#### Unity Catalog organiza todo como catalogo.schema.objeto. 
#### Esto para organizar los datos que se usaron para entrenar el modelo.


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.ml")
display(spark.sql("SHOW SCHEMAS IN workspace"))


databaseName
default
information_schema
ml
retail_mlops


#### Convertir datos.csv (archivo del workspace) en una tabla Delta de Unity Catalog.


In [0]:
import pandas as pd

RUTA_CSV = "/Workspace/deploy_ml_model/datos.csv"   # <- ajusta a tu carpeta (menu "..." > Copy URL/path)
pdf = pd.read_csv(RUTA_CSV)

# Delta no admite espacios en nombres de columna ("mean radius" -> "mean_radius").
pdf.columns = [c.replace(" ", "_") for c in pdf.columns]

spark.createDataFrame(pdf).write.mode("overwrite").saveAsTable("workspace.ml.breast_cancer")

df = spark.table("workspace.ml.breast_cancer")
print(df.count(), len(df.columns))      # esperado: 569 31
display(df.limit(5))

569 31


mean_radius,mean_texture,mean_perimeter,mean_area,mean_smoothness,mean_compactness,mean_concavity,mean_concave_points,mean_symmetry,mean_fractal_dimension,radius_error,texture_error,perimeter_error,area_error,smoothness_error,compactness_error,concavity_error,concave_points_error,symmetry_error,fractal_dimension_error,worst_radius,worst_texture,worst_perimeter,worst_area,worst_smoothness,worst_compactness,worst_concavity,worst_concave_points,worst_symmetry,worst_fractal_dimension,target
17.99,10.38,122.8,1001.0,0.1184,0.2776,0.3001,0.1471,0.2419,0.07871,1.095,0.9053,8.589,153.4,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.1189,0
20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.0186,0.0134,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.186,0.275,0.08902,0
19.69,21.25,130.0,1203.0,0.1096,0.1599,0.1974,0.1279,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.00615,0.04006,0.03832,0.02058,0.0225,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.243,0.3613,0.08758,0
11.42,20.38,77.58,386.1,0.1425,0.2839,0.2414,0.1052,0.2597,0.09744,0.4956,1.156,3.445,27.23,0.00911,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.5,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.173,0
20.29,14.34,135.1,1297.0,0.1003,0.1328,0.198,0.1043,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.01149,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.2,1575.0,0.1374,0.205,0.4,0.1625,0.2364,0.07678,0


#### Instalar la misma version de scikit-learn con la que se entreno el .pkl local.
#### Esta celda reinicia Python: ejecutala sola y antes de cargar el modelo.


In [0]:
%pip install scikit-learn==1.9.0 joblib
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 77.8 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.7.2
    Not uninstalling scikit-learn at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e154674f-6fec-4e27-8ac8-f35058d01320
    Can't uninstall 'scikit-learn'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


#### Cargar el modelo .pkl y probarlo con datos de la tabla.

In [0]:
import joblib

modelo = joblib.load("/Workspace/deploy_ml_model/modelo.pkl")

pdf = spark.table("workspace.ml.breast_cancer").toPandas()
X = pdf.drop(columns="target")
y = pdf["target"]
print(modelo.predict(X.head(5)), y.head(5).tolist())   
# las dos listas deben coincidir
# Si falla con "feature names should match": el modelo se entreno con nombres distintos
# a los de la tabla (espacios vs guion bajo). Reentrena localmente con los mismos nombres.



[0 0 0 0 0] [0, 0, 0, 0, 0]


#### Registrar el modelo en MLflow y Unity Catalog (version 1).


In [0]:
import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="cancer_rf") as run:
    preds = modelo.predict(X)
    mlflow.log_param("modelo", "RandomForest + StandardScaler")
    mlflow.log_metric("accuracy", float((preds == y).mean()))   # sobre los datos de entrenamiento: solo informativo
    info = mlflow.sklearn.log_model(
        modelo,
        artifact_path="model",       # en MLflow 3 tambien se acepta name="model"
        signature=infer_signature(X, preds),
        input_example=X.head(3),
        registered_model_name="workspace.ml.cancer_classifier",
    )
print(info.model_uri)


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6528463269380239>, line 7
      4 mlflow.set_registry_uri("databricks-uc")
      6 with mlflow.start_run(run_name="cancer_rf") as run:
----> 7     preds = modelo.predict(X)
      8     mlflow.log_param("modelo", "RandomForest + StandardScaler")
      9     mlflow.log_metric("accuracy", float((preds == y).mean()))   # sobre los datos de entrenamiento: solo informativo

NameError: name 'modelo' is not defined

#### Asignar el alias "champion" a la version 1.
#### El alias es un puntero con nombre: para promover una version nueva basta mover el alias.

In [0]:
from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
client.set_registered_model_alias("workspace.ml.cancer_classifier", "champion", 1)

mv = client.get_model_version_by_alias("workspace.ml.cancer_classifier", "champion")
print(mv.version, mv.aliases)           # esperado: 1 ['champion']

1 ['champion']


#### PASO MANUAL EN LA INTERFAZ (no hay codigo): crear el endpoint de Model Serving.
####   AI/ML > Serving > Create serving endpoint
#####   - Name:    cancer-classifier
#####   - Entity:  "My models" > workspace.ml.cancer_classifier > version 1
#####   - Compute: CPU (4 GB), concurrencia minima 0 (scale to zero), maxima 4
#####   - Dejar apagados: tracing y route optimization
####   Esperar de 5 a 15 min hasta que el estado sea "Ready".
#### Prueba rapida en la UI: boton "Use > Query".


#### Consumir el endpoint por codigo, como lo haria una aplicacion.

In [0]:
import mlflow.deployments

cliente = mlflow.deployments.get_deploy_client("databricks")

pdf = spark.table("workspace.ml.breast_cancer").toPandas()
X_api = pdf.drop(columns="target").head(5)

resp = cliente.predict(
    endpoint="cancer-classifier",
    inputs={"dataframe_split": X_api.to_dict(orient="split")},
)
print("endpoint:", resp["predictions"])
print("real:    ", pdf["target"].head(5).tolist())

{"ts": "2026-10-06 23:59:55.169", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzMxMDI1MzU4NTAyNDU3NBABIAEyJDAxYTExM2E4LTdlMWMtNzNhYi05ZTdlLWUwY2E5YzkxMTkyZTokYWM3M2QxOTQtZWM5Yi0zNDVjLTllNjgtODc3MjU5ZTk0NzMwSgwI45aW1gYQgMLynQNQAVgBYAFot9HJwdC61AOIAQA=.", "context": {}}
{"ts": "2026-10-06 23:59:55.169", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzMxMDI1MzU4NTAyNDU3NBABIAEyJDAxYTExM2E4LTdlMWMtNzNhYi05ZTdlLWUwY2E5YzkxMTkyZTokYWM3M2QxOTQtZWM5Yi0zNDVjLTllNjgtODc3MjU5ZTk0NzMwSgwI45aW1gYQgMLynQNQAVgBYAFot9HJwdC61AOIAQA=.", "context": {}}
{"ts": "2026-10-06 23:59:55.169", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvMzMxMDI1MzU4NTAyNDU3NBABIAEyJDAxYTExM2E4LTdlMWMtNzNhYi05ZTdlLWUwY2E5YzkxMTkyZTokYWM3M2QxOTQtZWM5Yi0zNDVjLTllNjgtODc3MjU5

endpoint: [0, 0, 0, 0, 0]
real:     [0, 0, 0, 0, 0]


## Feature Store Engineering:

In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.8 MB/s  0:00:00
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.122.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-d9b9cbd6-2285-4d1e-b0d9-bce00b5474fa
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [databricks-feature-engineering]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


#### Feature Store, paso 2: separar features y etiquetas, y crear la feature table.
#### Hace falta una llave primaria: la creamos como patient_id.

In [0]:
from pyspark.sql import functions as F, Window
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup

df = spark.table("workspace.ml.breast_cancer")
w = Window.orderBy(F.monotonically_increasing_id())
df = df.withColumn("patient_id", F.row_number().over(w))   # aviso de "no partition": inofensivo con pocos datos

features = df.drop("target")
labels = df.select("patient_id", "target")
labels.write.mode("overwrite").saveAsTable("workspace.ml.cancer_labels")

fe = FeatureEngineeringClient()
fe.create_table(
    name="workspace.ml.cancer_features",
    primary_keys=["patient_id"],
    df=features,
    description="Features de tumores (Breast Cancer Wisconsin)",
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


<FeatureTable: name='workspace.ml.cancer_features', table_id='ca329f45-9bed-4227-9564-5b6fe293360e', description='Features de tumores (Breast Cancer Wisconsin)', primary_keys=['patient_id'], partition_columns=[], features=['mean_radius',
 'mean_texture',
 'mean_perimeter',
 'mean_area',
 'mean_smoothness',
 'mean_compactness',
 'mean_concavity',
 'mean_concave_points',
 'mean_symmetry',
 'mean_fractal_dimension',
 'radius_error',
 'texture_error',
 'perimeter_error',
 'area_error',
 'smoothness_error',
 'compactness_error',
 'concavity_error',
 'concave_points_error',
 'symmetry_error',
 'fractal_dimension_error',
 'worst_radius',
 'worst_texture',
 'worst_perimeter',
 'worst_area',
 'worst_smoothness',
 'worst_compactness',
 'worst_concavity',
 'worst_concave_points',
 'worst_symmetry',
 'worst_fractal_dimension',
 'patient_id'], creation_timestamp=1791325575015, online_stores=[], notebook_producers=[], job_producers=[], table_data_sources=[], path_data_sources=[], custom_data_sources

#### Feature Store, paso 3: armar el training set uniendo etiquetas y features por llave.


In [0]:
lookups = [FeatureLookup(table_name="workspace.ml.cancer_features", lookup_key="patient_id")]
training_set = fe.create_training_set(
    df=labels, feature_lookups=lookups, label="target", exclude_columns=["patient_id"]
)
train_pdf = training_set.load_df().toPandas()
print(train_pdf.shape) 

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


(569, 31)


#### Entrenar con el training set y registrar un modelo ligado a la Feature Store.
#### Se registra con otro nombre para no tocar el modelo que ya sirve el endpoint.

In [0]:
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

mlflow.set_registry_uri("databricks-uc")

X = train_pdf.drop(columns="target")
y = train_pdf["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo_fs = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42)),
])

with mlflow.start_run(run_name="cancer_rf_featurestore"):
    modelo_fs.fit(X_train, y_train)
    auc = roc_auc_score(y_test, modelo_fs.predict_proba(X_test)[:, 1])
    mlflow.log_metric("roc_auc_test", auc)
    fe.log_model(
        model=modelo_fs,
        artifact_path="model",
        flavor=mlflow.sklearn,
        training_set=training_set,
        registered_model_name="workspace.ml.cancer_classifier_fs",
    )
print("ROC-AUC test:", round(auc, 3))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
🔗 View Logged Model at: https://dbc-55729a48-20c5.cloud.databricks.com/ml/experiments/3310253585024574/models/m-7ebb807983a84b89a837628960b6e4b8?o=2060300244379831
/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3343: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(
Successfully registered model 'workspace.ml.cancer_classifier_fs'.
2026/10/07 16:14:17 WARNING mlflow.tracking._model_registry.fluent: Run with id 0f322471d38b4142b8e12a

Uploading artifacts:   0%|          | 0/15 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.ml.cancer_classifier_fs': https://dbc-55729a48-20c5.cloud.databricks.com/explore/data/models/workspace/ml/cancer_classifier_fs/version/1?o=2060300244379831


ROC-AUC test: 0.994
